# Reproducing market making and electricity price formation in Japan

This notebook is the readable entry point for a public-data replication of Kanamura and Bunn (2022). It first checks the descriptive evidence behind the paper, before attempting the state-switching model.

**Question.** Did JEPX buy and sell bid volumes become larger and more balanced after gross bidding began in FY2017?

## How to verify this notebook

The processed analysis panel is committed with the repository. To rebuild it from public source files, run the next cell and then execute `build_panel.py`. The raw source files are intentionally not committed because they are larger and are reproducibly downloadable.

In [1]:
from pathlib import Path
import subprocess
import sys

slug = 'kanamura-bunn-2022-jepx'
project_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / 'replications' / slug,
    Path.cwd() / 'paper-replications' / 'replications' / slug,
    Path('/content/paper-replications/replications') / slug,
]
project = next((path for path in project_candidates if (path / 'scripts' / 'download_data.py').is_file()), None)
if project is None:
    raise FileNotFoundError('Could not locate the replication project directory.')

subprocess.run([sys.executable, str(project / 'scripts' / 'download_data.py')], check=True, cwd=project)

Saved data/raw/jepx_spot.csv
Saved data/raw/japan_weather.csv


## Build the panel

The sample spans 2015-04-01 through 2020-03-08. Market tightness is defined as buy bid volume minus sell bid volume. The paper uses Tokyo temperature as a first-order national proxy; this first stage uses the midpoint of daily maximum and minimum Tokyo temperature, explicitly as an approximation.

In [2]:
subprocess.run([sys.executable, str(project / 'scripts' / 'build_panel.py')], check=True, cwd=project)

Wrote 86,592 half-hourly observations to data/processed/kanamura_bunn_2022_panel.csv


## Descriptive result

The next cell regenerates the figures and fiscal-year statistics. The embedded outputs below are intentionally committed so that GitHub's notebook preview shows the result without running code.

In [3]:
subprocess.run([sys.executable, str(project / 'scripts' / 'descriptive_analysis.py')], check=True, cwd=project)

Wrote report/descriptive_statistics.csv
Wrote report/first_stage_results.md
Wrote figures/figure_01_descriptive_time_series.svg
Wrote figures/figure_02_temperature_scarcity.svg


### Fiscal-year summary

| Fiscal year | Mean price (JPY/kWh) | Mean buy bids (million kWh) | Mean sell bids (million kWh) | Mean tightness (million kWh) |
| --- | ---: | ---: | ---: | ---: |
| FY2015 | 9.777 | 2.110 | 5.063 | -2.953 |
| FY2016 | 8.463 | 3.474 | 5.529 | -2.055 |
| FY2017 | 9.719 | 6.528 | 7.941 | -1.413 |
| FY2018 | 9.756 | 15.470 | 15.681 | -0.211 |
| FY2019 | 8.067 | 20.065 | 19.885 | 0.181 |

The central descriptive pattern is clear: both sides of the market grew substantially and became more balanced. Price means alone do not establish the mechanism, which is why the original paper uses a nonlinear regime-switching model.

### Figure 1: Daily descriptive time series

![Four-panel JEPX descriptive time series](../figures/figure_01_descriptive_time_series.svg)

### Figure 2: Temperature and market tightness

![Temperature and tightness scatter plots](../figures/figure_02_temperature_scarcity.svg)

## What has and has not been reproduced

**Reproduced so far:** the paper's data logic, sample construction, market-tightness variable, and the descriptive claim that market activity rose after FY2017.

**Not yet claimed:** exact temperature input equivalence, state-switching parameter estimates, or a causal effect of gross bidding. The next research step is a piecewise V-shape regression of temperature against tightness, followed by sensitivity checks using alternative temperature constructions.